> **Note:** originally developed and run in Google Colab. Google-Drive mounting cells were removed, Drive paths were rewritten as relative paths and outputs were stripped (figures of the original run are in `assets/`). The paths/cleanup were **not re-executed** after this change, so expect to adjust paths.

#**LUAD**

In [ ]:
clinical = "data/TCGA-LUAD.clinical.tsv.gz"
star = "data/TCGA-LUAD.star_tpm.tsv.gz"
survival = "data/TCGA-LUAD.survival.tsv.gz"

In [ ]:
import pandas as pd
import numpy as np

expr = pd.read_csv(
    star,
    sep="\t",
    index_col=0
)

expr.shape


In [ ]:
surv = pd.read_csv(
    survival,
    sep="\t"
)

surv.columns

In [ ]:
pheno = pd.read_csv(
    clinical,
    sep="\t"
)

pheno.columns

In [ ]:
expr.columns = expr.columns.str.slice(0, 12)
surv["sample_id"] = surv["sample"].str.slice(0, 12)
pheno["sample_id"] = pheno["sample"].str.slice(0, 12)

In [ ]:
common_ids = (
    set(expr.columns)
    & set(surv["sample_id"])
    & set(pheno["sample_id"])
)

len(common_ids)

In [ ]:
expr_filt = expr[list(common_ids)]
surv_filt = surv[surv["sample_id"].isin(common_ids)]
pheno_filt = pheno[pheno["sample_id"].isin(common_ids)]

#**EDA**

In [ ]:
print("Expresión:", expr_filt.shape)
print("Survival:", surv_filt.shape)
print("Phenotype:", pheno_filt.shape)

In [ ]:
surv_filt["OS"].value_counts()

In [ ]:
surv_filt["OS.time"].describe()

In [ ]:
clinical_cols = [
    "sample_id",
    "OS.time",
    "OS",
    "age_at_diagnosis.diagnoses",
    "gender.demographic",
    "ajcc_pathologic_stage.diagnoses"
]

clin = (
    surv_filt.merge(pheno_filt, on="sample_id")
    [clinical_cols]
)

In [ ]:
clin.head()

In [ ]:
clin.isna().sum()

In [ ]:
#Este código elimina genes que prácticamente no se expresan, dejando solo aquellos con señal biológica real, mejorando la calidad del análisis posterior.
gene_means = expr_filt.mean(axis=1)
expr_filtered = expr_filt.loc[gene_means > 1]
expr_filtered.shape

In [ ]:
clin["OS"].value_counts()

In [ ]:
clin["OS.time"].describe()

In [ ]:
import matplotlib.pyplot as plt

plt.hist(clin["OS.time"], bins=50)
plt.xlabel("Overall survival time (days)")
plt.ylabel("Number of patients")
plt.title("Distribution of overall survival time")
plt.show()

In [ ]:
clin["age_at_diagnosis.diagnoses"].describe()

In [ ]:
clin["age_years"] = clin["age_at_diagnosis.diagnoses"] / 365.25
clin["age_years"].describe()

In [ ]:
plt.hist(clin["age_years"], bins=30)
plt.xlabel("Age at diagnosis")
plt.ylabel("Number of patients")
plt.title("Age distribution")
plt.show()

In [ ]:
clin[["age_years", "OS.time"]].corr()

In [ ]:
clin["age_group"] = pd.qcut(
    clin["age_at_diagnosis.diagnoses"],
    q=4,
    labels=["Q1 (young)", "Q2", "Q3", "Q4 (old)"]
)
clin["age_group"].value_counts()


In [ ]:
clin["gender.demographic"].value_counts()

In [ ]:
clin["ajcc_pathologic_stage.diagnoses"].value_counts(dropna=False)

In [ ]:
def simplify_stage(stage):
    if pd.isna(stage):
        return np.nan

    stage = stage.strip()

    if stage.startswith("Stage III") or stage.startswith("Stage IV"):
        return "Advanced"

    if stage.startswith("Stage I") or stage.startswith("Stage II"):
        return "Early"

    return np.nan


clin["stage_group"] = clin["ajcc_pathologic_stage.diagnoses"].apply(simplify_stage)
clin["stage_group"].value_counts(dropna=False)

In [ ]:
clin.groupby("stage_group")["OS.time"].median()

In [ ]:
clinical_summary = clin.groupby("stage_group").agg(
    n_patients=("sample_id", "count"),
    median_age=("age_years", "median"),
    deaths=("OS", "sum"),
    median_OS=("OS.time", "median")
)

clinical_summary

In [ ]:
stage_col = "ajcc_pathologic_stage.diagnoses"

summary_stage = (
    clin
    .dropna(subset=[stage_col])
    .groupby([ "stage_group", stage_col ])
    .agg(
        n_patients=("sample_id", "count"),
        events=("OS", "sum"),
        median_OS=("OS.time", "median")
    )
)

summary_stage["event_rate_pct"] = (
    summary_stage["events"] / summary_stage["n_patients"] * 100
).round(1)

In [ ]:
stage_order = [
    "Stage IA", "Stage IB",
    "Stage IIA", "Stage IIB",
    "Stage IIIA", "Stage IIIB",
    "Stage IV"
]

summary_stage = summary_stage.reset_index()

summary_stage[stage_col] = pd.Categorical(
    summary_stage[stage_col],
    categories=stage_order,
    ordered=True
)

summary_stage = (
    summary_stage
    .sort_values(["stage_group", stage_col])
    .set_index(["stage_group", stage_col])
    [["n_patients", "events", "event_rate_pct", "median_OS"]]
)


In [ ]:
summary_stage

In [ ]:
!pip install lifelines
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

kmf = KaplanMeierFitter()

In [ ]:
for group in clin["stage_group"].dropna().unique():
    ix = clin["stage_group"] == group
    kmf.fit(
        clin.loc[ix, "OS.time"],
        event_observed=clin.loc[ix, "OS"],
        label=group
    )
    kmf.plot_survival_function()

plt.title("Kaplan–Meier by stage group")
plt.xlabel("Days")
plt.ylabel("Survival probability")
plt.show()

El gráfico de Kaplan–Meier estratificado por grupo de estadio patológico muestra una separación clara y sostenida entre las curvas de supervivencia correspondientes a pacientes con enfermedad en estadio temprano (Early) y avanzado (Advanced) a lo largo de todo el período de seguimiento. Desde los primeros meses posteriores al diagnóstico se observa un descenso más pronunciado de la probabilidad de supervivencia en el grupo Advanced, lo que indica una mayor tasa de eventos tempranos en comparación con el grupo Early. Esta divergencia temprana sugiere que el estadio patológico al momento del diagnóstico es un determinante fuerte del pronóstico en adenocarcinoma de pulmón, incluso antes de considerar otras covariables clínicas o moleculares.

A medida que transcurre el tiempo de seguimiento, la distancia entre ambas curvas se mantiene e incluso se amplía, evidenciando diferencias sustanciales en la dinámica de supervivencia a largo plazo. Mientras que los pacientes en estadio temprano presentan una disminución más gradual de la supervivencia, con una proporción considerable de individuos que permanecen vivos durante varios años, el grupo Advanced alcanza probabilidades de supervivencia marcadamente bajas en un intervalo temporal más corto. Este patrón es consistente con la biología clínica de la enfermedad, donde los estadios avanzados suelen asociarse a mayor carga tumoral, presencia de enfermedad locorregional o metastásica y, en consecuencia, menor respuesta terapéutica y mayor riesgo de mortalidad.

Las bandas de confianza alrededor de las curvas reflejan la incertidumbre asociada a las estimaciones de supervivencia y se ensanchan progresivamente en tiempos de seguimiento más prolongados, particularmente en el grupo Advanced. Este comportamiento es esperable debido a la reducción del número de pacientes en riesgo a medida que ocurren eventos y censuras. No obstante, incluso considerando esta incertidumbre, la separación entre ambos grupos permanece claramente definida, lo que refuerza la robustez de la asociación entre estadio patológico y supervivencia global.

En conjunto, este análisis confirma que la estratificación por estadio patológico discrimina eficazmente subpoblaciones con pronósticos claramente diferenciados en la cohorte TCGA-LUAD. La marcada diferencia en las curvas de Kaplan–Meier valida el estadio como un “ground truth” clínico del riesgo de mortalidad y proporciona un marco de referencia sólido para análisis posteriores. En particular, estos resultados justifican la integración de variables transcriptómicas con el objetivo de identificar firmas moleculares que capturen, y potencialmente refinen, el riesgo pronóstico más allá de lo explicado por el estadio clínico convencional.


In [ ]:
cox_df = clin[[
    "OS.time",
    "OS",
    "age_years",
    "stage_group",
    "gender.demographic"
]].dropna()

cox_df["stage_group"] = (cox_df["stage_group"] == "Advanced").astype(int)
cox_df["gender_female"] = (cox_df["gender.demographic"] == "female").astype(int)

cox_df = cox_df.drop(columns=["gender.demographic"])

from lifelines import CoxPHFitter

cph = CoxPHFitter()
cph.fit(cox_df, duration_col="OS.time", event_col="OS")
cph.print_summary()

El modelo de riesgos proporcionales de Cox ajustado sobre la cohorte TCGA-LUAD permite cuantificar de manera multivariada el efecto de variables clínicas fundamentales —edad al diagnóstico, estadio patológico y sexo— sobre la supervivencia global. El análisis se realizó sobre 878 observaciones, de las cuales se registraron 334 eventos, lo que proporciona una base adecuada para la estimación estable de los coeficientes del modelo. El test global de razón de verosimilitud es altamente significativo, lo que indica que el conjunto de covariables incluidas mejora de forma sustancial el ajuste del modelo en comparación con un modelo nulo sin predictores.

El estadio patológico emerge como el principal determinante pronóstico en el modelo. La variable binaria que compara enfermedad avanzada frente a temprana presenta un coeficiente positivo elevado, correspondiente a un hazard ratio de aproximadamente 2.4, con un intervalo de confianza del 95% que no incluye la unidad y un valor de p extremadamente bajo. Esto implica que, manteniendo constantes la edad y el sexo, los pacientes diagnosticados en estadios avanzados presentan más del doble de riesgo instantáneo de muerte en comparación con aquellos en estadios tempranos. Este resultado es coherente con la evidencia clínica y valida el estadio como un fuerte factor pronóstico independiente en el adenocarcinoma de pulmón.

#**EDA Transcriptomico**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ---------------------------
# 0) Deduplicación clínica robusta (NO volver a redefinir clin_1 después)
# ---------------------------
key_cols = ["OS.time", "OS", "age_years", "stage_group"]
missing = [c for c in key_cols if c not in clin.columns]
if missing:
    raise ValueError(f"Faltan columnas en clin: {missing}")

clin_tmp = clin.copy()
clin_tmp["na_key"] = clin_tmp[key_cols].isna().sum(axis=1)

clin_1 = (
    clin_tmp
    .sort_values(by=["sample_id", "na_key", "OS.time"], ascending=[True, True, False])
    .drop_duplicates(subset=["sample_id"], keep="first")
    .drop(columns=["na_key"])
)

clin_idx = clin_1.set_index("sample_id")

## ---------------------------
# 1) Expresión única + genes variables + X (log1p)
# ---------------------------
expr_unique = expr_filtered.groupby(expr_filtered.columns, axis=1).mean()
gene_vars = expr_unique.var(axis=1)
top_var_genes = gene_vars.sort_values(ascending=False).head(2000).index

X = np.log1p(expr_unique.loc[top_var_genes].T)

# ---------------------------
# 2) PCA (solo para explorar estructura; usa stage_group si está)
# ---------------------------
common_ids = X.index.intersection(clin_idx.index)
X = X.loc[common_ids]
stage = clin_idx.reindex(common_ids)["stage_group"]

mask = stage.notna()
X_pca = X.loc[mask]
stage_pca = stage.loc[mask]

X_scaled = StandardScaler().fit_transform(X_pca)
pca = PCA(n_components=10, random_state=42)
pcs = pca.fit_transform(X_scaled)

evr = pca.explained_variance_ratio_
print("Explained variance ratio (PC1..PC10):", np.round(evr, 4))
print("Cumulative explained variance (PC1+PC2):", round(evr[:2].sum(), 4))

pca_df = pd.DataFrame(pcs[:, :3], index=X_pca.index, columns=["PC1", "PC2", "PC3"])
pca_df["stage_group"] = stage_pca.values

plt.figure(figsize=(6,5))
for g in ["Early", "Advanced"]:
    m = pca_df["stage_group"] == g
    plt.scatter(pca_df.loc[m, "PC1"], pca_df.loc[m, "PC2"], alpha=0.7, label=g)
plt.title("PCA (top variable genes) colored by stage_group")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.show()

In [ ]:
# 9) Medida simple de "señal clínica" en PCA:
# diferencia de medias en PC1/PC2 entre Early vs Advanced (tamaño de efecto de Cohen d)
def cohens_d(a, b):
    a = np.asarray(a); b = np.asarray(b)
    na, nb = len(a), len(b)
    sa, sb = a.std(ddof=1), b.std(ddof=1)
    sp = np.sqrt(((na-1)*sa**2 + (nb-1)*sb**2) / (na+nb-2))
    return (a.mean() - b.mean()) / sp

early = pca_df[pca_df["stage_group"] == "Early"]
adv   = pca_df[pca_df["stage_group"] == "Advanced"]

d_pc1 = cohens_d(early["PC1"], adv["PC1"])
d_pc2 = cohens_d(early["PC2"], adv["PC2"])

print("Cohen's d (Early - Advanced):")
print("PC1:", round(d_pc1, 3))
print("PC2:", round(d_pc2, 3))

# Interpretación rápida:
# |d| ~ 0.2 pequeño, ~0.5 moderado, ~0.8 grande

In [ ]:
# ---------------------------
# 3) Alinear clínica con X (para supervivencia)
# ---------------------------
clin_X = clin_idx.reindex(X.index)
print("NA en outcome:", clin_X[["OS.time", "OS"]].isna().sum())

base = clin_X[["OS.time", "OS", "age_years", "stage_group"]].dropna()
X_base = X.loc[base.index]

In [ ]:
from sklearn.model_selection import train_test_split
# ---------------------------
# 4) Train/Test split + screening en TRAIN
# ---------------------------
train_ids, test_ids = train_test_split(
    base.index,
    test_size=0.30,
    random_state=42,
    stratify=base["OS"]
)

X_train = X_base.loc[train_ids]
X_test  = X_base.loc[test_ids]

clin_train = base.loc[train_ids].copy()
clin_test  = base.loc[test_ids].copy()

results = []
for gene in X_train.columns:
    df = pd.DataFrame({
        "OS.time": clin_train["OS.time"],
        "OS": clin_train["OS"],
        "gene": X_train[gene]
    }).dropna()

    if df["gene"].var() == 0:
        continue

    cph = CoxPHFitter()
    cph.fit(df, duration_col="OS.time", event_col="OS")

    results.append({
        "gene": gene,
        "HR": float(np.exp(cph.params_["gene"])),
        "p": float(cph.summary.loc["gene", "p"])
    })

res_train = pd.DataFrame(results).sort_values("p").reset_index(drop=True)
display(res_train.head(10))

In [ ]:
# ---------------------------
# 5) Firma (top 15 genes TRAIN) + umbral TRAIN
# ---------------------------
signature_genes = res_train.head(15)["gene"].tolist()

clin_train["signature_score"] = X_train[signature_genes].mean(axis=1)
clin_test["signature_score"]  = X_test[signature_genes].mean(axis=1)

threshold = clin_train["signature_score"].median()
clin_test["signature_group"] = np.where(clin_test["signature_score"] >= threshold, "High risk", "Low risk")

In [ ]:
# ---------------------------
# 6) KM en TEST
# ---------------------------
kmf = KaplanMeierFitter()
plt.figure(figsize=(6,5))
for g in ["Low risk", "High risk"]:
    ix = clin_test["signature_group"] == g
    kmf.fit(clin_test.loc[ix, "OS.time"], clin_test.loc[ix, "OS"], label=g)
    kmf.plot_survival_function()
plt.title("Kaplan–Meier (TEST) by transcriptomic risk group")
plt.xlabel("Days")
plt.ylabel("Survival probability")
plt.show()

In [ ]:
# ---------------------------
# 7) Cox multivariado en TEST
# ---------------------------
cox_test = clin_test[["OS.time", "OS", "age_years", "stage_group", "signature_score"]].dropna().copy()
cox_test["stage_group"] = (cox_test["stage_group"] == "Advanced").astype(int)

cph_test = CoxPHFitter()
cph_test.fit(cox_test, duration_col="OS.time", event_col="OS")
cph_test.print_summary()

#**Interpretación de la firma**

In [ ]:
import numpy as np
import pandas as pd

# Asegurar que res_train tenga gene, HR, p
res_train_small = res_train[["gene", "HR", "p"]].copy()
res_train_small["logHR"] = np.log(res_train_small["HR"])

# Genes de firma con dirección
sig_df = res_train_small.set_index("gene").loc[signature_genes].reset_index()

risk_genes = sig_df.loc[sig_df["HR"] > 1, "gene"].tolist()
protective_genes = sig_df.loc[sig_df["HR"] < 1, "gene"].tolist()

print("Firma total:", len(signature_genes))
print("Riesgo (HR>1):", len(risk_genes))
print("Protectores (HR<1):", len(protective_genes))

display(sig_df.sort_values("p"))

In [ ]:
import matplotlib.pyplot as plt

sig_df_plot = sig_df.copy()
sig_df_plot["direction"] = np.where(sig_df_plot["HR"] > 1, "Risk-associated", "Protective")

# ordenar por |logHR|
sig_df_plot["abs_logHR"] = sig_df_plot["logHR"].abs()
sig_df_plot = sig_df_plot.sort_values("abs_logHR", ascending=True)

plt.figure(figsize=(7,6))
plt.barh(sig_df_plot["gene"], sig_df_plot["logHR"])
plt.axvline(0, linewidth=1)
plt.xlabel("log(HR) from univariate Cox (TRAIN)")
plt.title("Signature genes: direction of effect on survival")
plt.tight_layout()
plt.show()

In [ ]:
rank_df = res_train_small.dropna().copy()
rank_df = rank_df[rank_df["p"] > 0]  # por seguridad
rank_df["score"] = np.sign(rank_df["logHR"]) * (-np.log10(rank_df["p"]))

# (opcional) cap para evitar outliers extremos por p muy chica
rank_df["score"] = rank_df["score"].clip(lower=-50, upper=50)

# Orden descendente: arriba genes de riesgo fuerte, abajo protectores fuertes
rank_df = rank_df.sort_values("score", ascending=False)

# Formato típico para gseapy: dos columnas [gene, score]
rnk = rank_df[["gene", "score"]]

print("Ranking genes:", rnk.shape)
display(rnk.head(10))
display(rnk.tail(10))

In [ ]:
!pip -q install mygene

import re
import pandas as pd
import numpy as np
import mygene

def strip_ensembl_version(x: str) -> str:
    # ENSG00000123456.12 -> ENSG00000123456
    return re.sub(r"\.\d+$", "", str(x))

# 1) Preparar listado de Ensembl sin versión desde tu ranking
ens = rnk["gene"].astype(str).map(strip_ensembl_version).unique().tolist()

mg = mygene.MyGeneInfo()

# 2) Query masivo (Human)
q = mg.querymany(
    ens,
    scopes="ensembl.gene",
    fields="symbol",
    species="human",
    as_dataframe=True
)

# 3) Construir mapa ENSG -> SYMBOL
# q index = query (ENSG), columna 'symbol'
ens2sym = q["symbol"].dropna().to_dict()

print("Total IDs:", len(ens))
print("Mapeados a símbolo:", len(ens2sym))

# 4) Aplicar mapa al ranking
rnk_sym = rnk.copy()
rnk_sym["ensembl"] = rnk_sym["gene"].astype(str).map(strip_ensembl_version)
rnk_sym["symbol"] = rnk_sym["ensembl"].map(ens2sym)

# filtrar no mapeados + duplicados por símbolo (quedarse con el score más extremo)
rnk_sym = rnk_sym.dropna(subset=["symbol"]).copy()

# En caso de genes repetidos (múltiples ENSG al mismo símbolo), quedarse con el score de mayor |score|
rnk_sym["abs_score"] = rnk_sym["score"].abs()
rnk_sym = (
    rnk_sym.sort_values("abs_score", ascending=False)
           .drop_duplicates(subset=["symbol"], keep="first")
           .sort_values("score", ascending=False)
)

# Formato final para prerank
rnk_for_gsea = rnk_sym[["symbol", "score"]].rename(columns={"symbol":"gene"})

print("Ranking final (símbolos):", rnk_for_gsea.shape)
display(rnk_for_gsea.head(10))

In [ ]:
!pip -q install gseapy
import gseapy as gp

pre_res = gp.prerank(
    rnk=rnk_for_gsea,
    gene_sets="MSigDB_Hallmark_2020",
    min_size=15,
    max_size=500,
    permutation_num=200,
    seed=42,
    outdir=None,
    verbose=True
)

gsea_res = pre_res.res2d.sort_values("FDR q-val")
display(gsea_res.head(15))

In [ ]:
import matplotlib.pyplot as plt

top = gsea_res.head(12).copy()
top = top.iloc[::-1]

plt.figure(figsize=(8,6))
plt.barh(top.index, top["NES"])
plt.axvline(0, linewidth=1)
plt.xlabel("NES (GSEA preranked)")
plt.title("Top Hallmark pathways enriched in LUAD survival ranking")
plt.tight_layout()
plt.show()


In [ ]:
# Convertir firma a símbolos usando el mismo mapa
sig_ens = [strip_ensembl_version(g) for g in signature_genes]
sig_sym = [ens2sym.get(g, None) for g in sig_ens]
sig_sym = [s for s in sig_sym if s is not None]

print("Firma original:", len(signature_genes))
print("Firma mapeada a símbolos:", len(sig_sym))
print(sig_sym)

ora = gp.enrichr(
    gene_list=sig_sym,
    gene_sets=["MSigDB_Hallmark_2020", "GO_Biological_Process_2021", "KEGG_2021_Human"],
    organism="Human",
    outdir=None
)

for lib in ora.results["Gene_set"].unique():
    print("\n===", lib, "===")
    display(
        ora.results[ora.results["Gene_set"] == lib]
        .sort_values("Adjusted P-value")
        .head(10)[["Term", "Adjusted P-value", "Overlap", "Genes"]]
    )

In [ ]:
# Tomamos Hallmark del prerank como "take-home message"
top_tbl = (
    gsea_res
    .reset_index()
    .rename(columns={"index": "Pathway"})
    [["Pathway", "NES", "FDR q-val", "Lead_genes"]]
    .head(8)
)

display(top_tbl)

In [ ]:
gsea_res.head(10)

In [ ]:
sig_df[["gene", "HR", "p"]]

**🔬 GSEA preranked: la señal dominante es proliferativa y mesenquimal**

El análisis GSEA sobre el ranking transcriptómico asociado a supervivencia muestra un patrón muy claro. Las vías más enriquecidas corresponden a procesos centrales de progresión tumoral, con valores de NES elevados y FDR prácticamente nulos.

Las señales más fuertes están dominadas por G2/M checkpoint y E2F targets, lo que indica que los pacientes con peor pronóstico presentan una activación marcada de programas de proliferación celular y control del ciclo celular. Este hallazgo es consistente con una mayor fracción de células en división activa y con tumores biológicamente más agresivos.

En paralelo, aparece de forma robusta la transición epitelio-mesenquimal (EMT), una firma clásica asociada a invasión, plasticidad celular y potencial metastásico. La coexistencia de proliferación elevada y EMT sugiere que el transcriptoma está capturando no solo velocidad de crecimiento tumoral, sino también capacidad de diseminación.

Otras vías relevantes refuerzan este eje biológico: hipoxia, glucólisis y mTORC1 signaling apuntan a una reprogramación metabólica adaptativa, típica de tumores que crecen en microambientes hostiles. Finalmente, la activación de TNF-α / NF-κB introduce un componente inflamatorio y de señalización de estrés, frecuentemente vinculado a progresión tumoral y resistencia terapéutica.

En conjunto, el GSEA indica que el riesgo transcriptómico no es una señal abstracta, sino la manifestación coordinada de proliferación, plasticidad celular y adaptación metabólica, todos rasgos bien establecidos de mal pronóstico en LUAD.

**🧬 Firma transcriptómica: genes con roles funcionales coherentes**

La firma final de 15 genes es heterogénea, pero funcionalmente interpretable. Incluye genes asociados a:

* Remodelado de matriz extracelular e invasión (LOXL2, FLNC)

* Señalización Wnt / diferenciación (DKK1)

* Identidad epitelial y queratinización (KRT6A, KRT81)

* Componentes inmunes y linfocitarios (MS4A1, CD79A, CXCL17)

* Factores de desarrollo y regulación transcripcional (IRX3, IRX5)

El hecho de que algunos genes presenten HR > 1 y otros HR < 1 es esperable y deseable: la firma no representa un único proceso, sino un balance entre programas biológicos que favorecen o atenúan el riesgo. El puntaje transcriptómico emerge justamente de esa combinación.

Es importante notar que el análisis ORA sobre la firma individual no arroja enriquecimientos significativos tras corrección múltiple. Esto no es una debilidad, sino una consecuencia directa del tamaño reducido de la firma. La coherencia biológica aparece con claridad cuando se analiza el transcriptoma completo mediante GSEA, no cuando se evalúan 15 genes de forma aislada.



**📊 Coherencia global**

Hay una alineación muy sólida entre:

* GSEA (señal global del transcriptoma),

* firma transcriptómica (resumen parsimonioso),

* Kaplan–Meier en test,

* y modelo de Cox multivariado.

Todo apunta a que la firma está capturando un eje biológico real de agresividad tumoral, y no una asociación espuria.